# Convert clef-flash weights to an int8 JAX checkpoint for a v5e-1 TPU

Turns Cloudflare's BF16 release (`Cloudflare/clef-flash`, 18.8 GB) into safetensors that a JAX runtime can load into a v5e-1 TPU's 16 GB. This notebook converts and checks the weights only; it does not run the model.

The source is the original BF16 checkpoint, not `Cloudflare_clef-flash-Q4_K_M.gguf`: converting the Q4 file would carry its 4-bit rounding error into the int8 weights.

What changes:
- **Large projections** (attention, linear attention, MLP) become int8, symmetric, with one fp32 scale per output channel, so `w ≈ int8 * scale`. They are stored as `(in, out)`, so `x @ w` needs no transpose.
- **Same-type layers are stacked** on a leading layer axis, so the runtime can loop over them with `jax.lax.scan`. Each file's metadata lists the original layer indices.
- **Everything else keeps its exact original values and dtype:** norms, the conv, the gates, `A_log`, `dt_bias`, the embeddings and the LM head.
- **The vision tower is dropped**, since FIR input is text only.
- **Copied unchanged:** the decision head (`joint_head.safetensors`), the tokenizer, and Cloudflare's `joint_schema_model.py`.

**Runtime:** any Colab runtime with about 35 GB of free disk. Conversion runs on the CPU and streams one layer at a time, so it uses only a few GB of RAM. Use the TPU v5e-1 runtime if you'll load the model in the same session.

In [ ]:
import shutil
from pathlib import Path

REPO = "Cloudflare/clef-flash"
SOURCE_DIR = Path("/content/clef-flash")
OUTPUT_DIR = Path("/content/clef-flash-tpu")
# 18.8 GB download plus ~11 GB output, with headroom.
REQUIRED_FREE_GB = 35

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"{free_gb:.0f} GB free on /content")
assert free_gb >= REQUIRED_FREE_GB, f"need {REQUIRED_FREE_GB} GB free disk"

In [ ]:
%pip install -q -U huggingface_hub safetensors
from huggingface_hub import HfApi

# The TPU runtime ships no hf command and pip doesn't add one when the package is already installed,
# so use Hugging Face's standalone CLI installer, which links hf into ~/.local/bin.
if shutil.which("hf") is None:
    !curl -LsSf https://hf.co/cli/install.sh | bash -s -- --exclude-skill
HF_CLI = shutil.which("hf") or str(Path.home() / ".local/bin/hf")
assert Path(HF_CLI).exists(), f"hf CLI not found at {HF_CLI}; see the installer output above"
print(f"hf CLI: {HF_CLI}")

# Pinned to the commit resolved now, so the download and the recorded source match.
REVISION = HfApi().model_info(REPO).sha
print(f"{REPO} @ {REVISION}")
!{HF_CLI} download {REPO} --revision {REVISION} --local-dir {SOURCE_DIR}
!du -sh {SOURCE_DIR}

In [ ]:
import json
import struct
from dataclasses import dataclass
from typing import Callable, Iterator

import numpy as np

# numpy has no bfloat16, so BF16 tensors travel as their raw 16-bit patterns.
STORAGE_DTYPES = {"BF16": np.uint16, "F16": np.float16, "F32": np.float32, "I8": np.int8}
TEXT_PREFIX = "model.language_model."
VISION_PREFIX = "model.visual."
INT8_MAX = 127


class SafetensorsFile:
    def __init__(self, path):
        with open(path, "rb") as file:
            header_length = struct.unpack("<Q", file.read(8))[0]
            self.header = json.loads(file.read(header_length))
        self.metadata = self.header.pop("__metadata__", {})
        self.data_start = 8 + header_length
        self.bytes = np.memmap(path, dtype=np.uint8, mode="r")

    def raw(self, name):
        entry = self.header[name]
        start, end = entry["data_offsets"]
        data = self.bytes[self.data_start + start : self.data_start + end]
        return data.view(STORAGE_DTYPES[entry["dtype"]]).reshape(entry["shape"])


def to_float32(raw, dtype):
    if dtype == "BF16":
        return (raw.astype(np.uint32) << 16).view(np.float32)
    return raw.astype(np.float32)


class ShardedCheckpoint:
    def __init__(self, directory):
        index = json.loads((directory / "model.safetensors.index.json").read_text())
        self.weight_map = index["weight_map"]
        self.shards = {file: SafetensorsFile(directory / file) for file in set(self.weight_map.values())}
        self.read_names = set()

    def entry(self, name):
        return self.shards[self.weight_map[name]].header[name]

    def raw(self, name):
        self.read_names.add(name)
        return self.shards[self.weight_map[name]].raw(name)

    def float32(self, name):
        return to_float32(self.raw(name), self.entry(name)["dtype"])


def int8_scales(weight):
    absmax = np.abs(weight).max(axis=1)
    return np.where(absmax > 0, absmax / INT8_MAX, 1.0).astype(np.float32)


def quantize_int8(weight):
    """Quantizes an (out, in) weight and returns it as (in, out), the layout `x @ w` reads without a transpose."""
    quantized = np.rint(weight / int8_scales(weight)[:, None])
    return np.clip(quantized, -INT8_MAX, INT8_MAX).astype(np.int8).T


@dataclass
class OutputTensor:
    name: str
    dtype: str
    shape: tuple
    chunks: Callable[[], Iterator[np.ndarray]]


def layer_names(layers, suffix):
    return [f"{TEXT_PREFIX}layers.{layer}.{suffix}" for layer in layers]


def uniform_entry(checkpoint, names):
    entries = [checkpoint.entry(name) for name in names]
    first = entries[0]
    assert all(entry["dtype"] == first["dtype"] and entry["shape"] == first["shape"] for entry in entries), names[0]
    return first["dtype"], tuple(first["shape"])


def stacked_int8(checkpoint, key, names):
    _, (out_features, in_features) = uniform_entry(checkpoint, names)
    return [
        OutputTensor(
            f"{key}.int8", "I8", (len(names), in_features, out_features),
            lambda: (quantize_int8(checkpoint.float32(name)) for name in names),
        ),
        OutputTensor(
            f"{key}.scale", "F32", (len(names), out_features),
            lambda: (int8_scales(checkpoint.float32(name)) for name in names),
        ),
    ]


def stacked_raw(checkpoint, key, names, transpose=False):
    dtype, shape = uniform_entry(checkpoint, names)
    stored_shape = shape[::-1] if transpose else shape
    return [
        OutputTensor(
            key, dtype, (len(names), *stored_shape),
            lambda: (checkpoint.raw(name).T if transpose else checkpoint.raw(name) for name in names),
        )
    ]


def single_raw(checkpoint, key, name, rows_per_chunk=16384):
    dtype, shape = uniform_entry(checkpoint, [name])

    def chunks():
        raw = checkpoint.raw(name)
        for start in range(0, shape[0], rows_per_chunk):
            yield raw[start : start + rows_per_chunk]

    return [OutputTensor(key, dtype, shape, chunks)]


def write_safetensors(path, tensors, metadata):
    header = {"__metadata__": metadata}
    offset = 0
    for tensor in tensors:
        size = int(np.prod(tensor.shape, dtype=np.int64)) * np.dtype(STORAGE_DTYPES[tensor.dtype]).itemsize
        header[tensor.name] = {"dtype": tensor.dtype, "shape": list(tensor.shape), "data_offsets": [offset, offset + size]}
        offset += size
    header_bytes = json.dumps(header).encode()
    header_bytes += b" " * (-len(header_bytes) % 8)
    with open(path, "wb") as file:
        file.write(struct.pack("<Q", len(header_bytes)))
        file.write(header_bytes)
        for tensor in tensors:
            start, end = header[tensor.name]["data_offsets"]
            written = 0
            for chunk in tensor.chunks():
                chunk = np.ascontiguousarray(chunk, dtype=STORAGE_DTYPES[tensor.dtype])
                file.write(chunk.data)
                written += chunk.nbytes
            assert written == end - start, f"{tensor.name}: wrote {written} bytes, expected {end - start}"

In [ ]:
import time

config = json.loads((SOURCE_DIR / "config.json").read_text())
layer_types = config["text_config"]["layer_types"]
linear_layers = [layer for layer, kind in enumerate(layer_types) if kind == "linear_attention"]
full_layers = [layer for layer, kind in enumerate(layer_types) if kind == "full_attention"]
all_layers = list(range(len(layer_types)))
assert len(linear_layers) + len(full_layers) == len(layer_types), f"unknown layer types: {set(layer_types)}"

checkpoint = ShardedCheckpoint(SOURCE_DIR)


def int8_projections(layers, module, keys):
    return [tensor for key in keys for tensor in stacked_int8(checkpoint, key, layer_names(layers, f"{module}.{key}.weight"))]


def raw_per_layer(layers, keys_and_suffixes, transpose=False):
    return [
        tensor
        for key, suffix in keys_and_suffixes
        for tensor in stacked_raw(checkpoint, key, layer_names(layers, suffix), transpose=transpose)
    ]


output_files = {
    "linear_attention.safetensors": (linear_layers, [
        *int8_projections(linear_layers, "linear_attn", ["in_proj_qkv", "in_proj_z", "out_proj"]),
        *raw_per_layer(linear_layers, [("in_proj_a", "linear_attn.in_proj_a.weight"), ("in_proj_b", "linear_attn.in_proj_b.weight")], transpose=True),
        *raw_per_layer(linear_layers, [
            ("conv1d", "linear_attn.conv1d.weight"),
            ("A_log", "linear_attn.A_log"),
            ("dt_bias", "linear_attn.dt_bias"),
            ("norm", "linear_attn.norm.weight"),
        ]),
    ]),
    "full_attention.safetensors": (full_layers, [
        *int8_projections(full_layers, "self_attn", ["q_proj", "k_proj", "v_proj", "o_proj"]),
        *raw_per_layer(full_layers, [("q_norm", "self_attn.q_norm.weight"), ("k_norm", "self_attn.k_norm.weight")]),
    ]),
    "mlp.safetensors": (all_layers, int8_projections(all_layers, "mlp", ["gate_proj", "up_proj", "down_proj"])),
    "layer_norms.safetensors": (all_layers, [
        *raw_per_layer(all_layers, [("input_layernorm", "input_layernorm.weight"), ("post_attention_layernorm", "post_attention_layernorm.weight")]),
        *single_raw(checkpoint, "final_norm", f"{TEXT_PREFIX}norm.weight"),
    ]),
    "embeddings.safetensors": ([], single_raw(checkpoint, "embed_tokens", f"{TEXT_PREFIX}embed_tokens.weight")),
    "lm_head.safetensors": ([], single_raw(checkpoint, "lm_head", "lm_head.weight")),
}

QUANTIZATION = "int8 symmetric per output channel: w ~= int8 * scale, int8 stored as (in, out)"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for file_name, (layers, tensors) in output_files.items():
    started = time.time()
    metadata = {"source": f"{REPO}@{REVISION}", "layers": json.dumps(layers), "quantization": QUANTIZATION}
    write_safetensors(OUTPUT_DIR / file_name, tensors, metadata)
    size_gb = (OUTPUT_DIR / file_name).stat().st_size / 1e9
    print(f"{file_name}: {len(tensors)} tensors, {size_gb:.2f} GB, {time.time() - started:.0f}s")

text_names = {name for name in checkpoint.weight_map if not name.startswith(VISION_PREFIX)}
assert checkpoint.read_names == text_names, f"not converted: {sorted(text_names - checkpoint.read_names)[:5]}"
print(f"converted all {len(text_names)} text tensors; dropped {len(checkpoint.weight_map) - len(text_names)} vision tensors")

UNCHANGED_FILES = [
    "config.json", "joint_head.safetensors", "joint_head_config.json", "joint_schema_model.py",
    "tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "LICENSE",
]
for file_name in UNCHANGED_FILES:
    shutil.copy2(SOURCE_DIR / file_name, OUTPUT_DIR / file_name)

conversion = {
    "source": f"{REPO}@{REVISION}",
    "quantization": QUANTIZATION,
    "layers": {"linear_attention": linear_layers, "full_attention": full_layers},
    "files": {file_name: [tensor.name for tensor in tensors] for file_name, (_, tensors) in output_files.items()},
}
(OUTPUT_DIR / "conversion.json").write_text(json.dumps(conversion, indent=2))

Checks, reading the written files back:
- **int8 projections:** the first and last layer of each projection are multiplied by random activations, comparing the int8 result with the original BF16. Plain int8 rounding gives about 0.01 relative error. A wrong layout or scale gives an error near 1.
- **Unchanged tensors:** a sample is compared bit for bit with the source.
- **Format:** the official `safetensors` library opens every file and finds the expected tensors.

In [ ]:
from safetensors import safe_open

rng = np.random.default_rng(0)
converted = {file_name: SafetensorsFile(OUTPUT_DIR / file_name) for file_name in output_files}
source = ShardedCheckpoint(SOURCE_DIR)


def matmul_relative_error(weight, quantized, scales):
    activations = rng.standard_normal((32, weight.shape[1])).astype(np.float32)
    exact = activations @ weight.T
    approximate = (activations @ quantized.astype(np.float32)) * scales
    return float(np.linalg.norm(approximate - exact) / np.linalg.norm(exact))


INT8_CHECKS = [
    ("linear_attention.safetensors", linear_layers, "linear_attn", ["in_proj_qkv", "in_proj_z", "out_proj"]),
    ("full_attention.safetensors", full_layers, "self_attn", ["q_proj", "k_proj", "v_proj", "o_proj"]),
    ("mlp.safetensors", all_layers, "mlp", ["gate_proj", "up_proj", "down_proj"]),
]
worst_error = 0.0
for file_name, layers, module, keys in INT8_CHECKS:
    for key in keys:
        for position in (0, len(layers) - 1):
            weight = source.float32(f"{TEXT_PREFIX}layers.{layers[position]}.{module}.{key}.weight")
            error = matmul_relative_error(
                weight,
                converted[file_name].raw(f"{key}.int8")[position],
                converted[file_name].raw(f"{key}.scale")[position],
            )
            worst_error = max(worst_error, error)
            print(f"{module}.{key} layer {layers[position]}: relative matmul error {error:.4f}")
assert worst_error < 0.05, f"int8 error {worst_error:.3f} is far above rounding; layout or scale is wrong"

last_linear, last_layer = linear_layers[-1], all_layers[-1]
UNCHANGED_CHECKS = [
    ("linear_attention.safetensors", "in_proj_a", -1, f"layers.{last_linear}.linear_attn.in_proj_a.weight", True),
    ("linear_attention.safetensors", "A_log", -1, f"layers.{last_linear}.linear_attn.A_log", False),
    ("linear_attention.safetensors", "conv1d", 0, f"layers.{linear_layers[0]}.linear_attn.conv1d.weight", False),
    ("full_attention.safetensors", "q_norm", -1, f"layers.{full_layers[-1]}.self_attn.q_norm.weight", False),
    ("layer_norms.safetensors", "post_attention_layernorm", -1, f"layers.{last_layer}.post_attention_layernorm.weight", False),
]
for file_name, key, position, suffix, transposed in UNCHANGED_CHECKS:
    original = source.raw(f"{TEXT_PREFIX}{suffix}")
    assert np.array_equal(converted[file_name].raw(key)[position], original.T if transposed else original), key
assert np.array_equal(converted["embeddings.safetensors"].raw("embed_tokens")[-3:], source.raw(f"{TEXT_PREFIX}embed_tokens.weight")[-3:])
assert np.array_equal(converted["lm_head.safetensors"].raw("lm_head")[:3], source.raw("lm_head.weight")[:3])
assert np.array_equal(converted["layer_norms.safetensors"].raw("final_norm"), source.raw(f"{TEXT_PREFIX}norm.weight"))
print("unchanged tensors are bit-identical to the source")

for file_name in output_files:
    with safe_open(OUTPUT_DIR / file_name, framework="numpy") as file:
        assert set(file.keys()) == set(converted[file_name].header), file_name
        assert file.metadata()["source"] == f"{REPO}@{REVISION}", file_name
with safe_open(OUTPUT_DIR / "mlp.safetensors", framework="numpy") as file:
    assert np.array_equal(file.get_tensor("down_proj.scale"), converted["mlp.safetensors"].raw("down_proj.scale"))
print("all files open with the safetensors library")

In [ ]:
sizes_gb = {path.name: path.stat().st_size / 1e9 for path in sorted(OUTPUT_DIR.iterdir())}
for file_name, size_gb in sizes_gb.items():
    print(f"{size_gb:6.2f} GB  {file_name}")

# The decision head reads only the LM head rows of option tokens, so the full LM head can stay in host memory.
device_files = [file_name for file_name in output_files if file_name != "lm_head.safetensors"] + ["joint_head.safetensors"]
print(f"total {sum(sizes_gb.values()):.1f} GB; weights on the TPU: {sum(sizes_gb[file_name] for file_name in device_files):.1f} GB of 16 GB")

Push the converted folder (about 11 GB) to a private Hugging Face repo, so later sessions download it instead of converting again.

In [ ]:
from huggingface_hub import HfApi, get_token, login

# Uses the HF_TOKEN Colab secret if set; otherwise asks for a token with write access (huggingface.co/settings/tokens).
if get_token() is None:
    login()
api = HfApi()
TARGET_REPO = f"{api.whoami()['name']}/clef-flash-tpu-int8"
api.create_repo(TARGET_REPO, private=True, exist_ok=True)
# Resumable: re-running the cell after a disconnect uploads only the missing files.
api.upload_large_folder(repo_id=TARGET_REPO, folder_path=OUTPUT_DIR, repo_type="model")
print(f"uploaded to https://huggingface.co/{TARGET_REPO}")
print(f"next session: hf download {TARGET_REPO} --local-dir {OUTPUT_DIR}")

## Quality report: all int8 matrices

Every int8 matrix (200) against its BF16 original, as signal-to-noise of the matmul output in dB, where 40 dB is about 1% error:
- **40 dB or more:** excellent, rounding only.
- **30 to 40 dB:** good.
- **Below 30 dB:** worth trying block-wise scales.

The inputs are random. Real activations have a few large values, so real error can be somewhat higher. Run this in the same session as the conversion, because it rereads the source weights.

In [ ]:
import statistics
from collections import defaultdict

sqnr_by_projection = defaultdict(list)
for file_name, layers, module, keys in INT8_CHECKS:
    for key in keys:
        quantized_stack = converted[file_name].raw(f"{key}.int8")
        scale_stack = converted[file_name].raw(f"{key}.scale")
        for position, layer in enumerate(layers):
            weight = source.float32(f"{TEXT_PREFIX}layers.{layer}.{module}.{key}.weight")
            error = matmul_relative_error(weight, quantized_stack[position], scale_stack[position])
            sqnr_by_projection[f"{module}.{key}"].append((-20 * np.log10(error), layer))

print(f"{'projection':24} {'matrices':>8} {'median dB':>10} {'worst dB':>9} {'worst layer':>12}")
all_sqnr = []
for projection, results in sqnr_by_projection.items():
    values = [sqnr for sqnr, _ in results]
    worst_sqnr, worst_layer = min(results)
    all_sqnr += values
    print(f"{projection:24} {len(values):8} {statistics.median(values):10.1f} {worst_sqnr:9.1f} {worst_layer:12}")

worst = min(all_sqnr)
verdict = "excellent" if worst >= 40 else "good" if worst >= 30 else "worth trying block-wise scales"
print(f"\nall {len(all_sqnr)} int8 matrices: median {statistics.median(all_sqnr):.1f} dB, worst {worst:.1f} dB -> {verdict}")

## TPU latency: compute proxy

A stand-in workload that produces no answers; `hub/clef_tpu.py` runs the real model. This one runs the real model's workload on the converted weights at the T4 benchmark's size (6,820 tokens):
- all 32 layers' int8 projections and MLPs;
- causal attention in the 8 full-attention layers;
- a chunked linear-attention scan with Gated DeltaNet's shapes and its serial chunk steps, but simpler math.

Left out because they're cheap: the embedding lookup, rotary positions, the short convolution, the exact DeltaNet update and the decision head. Read the result as a close lower bound: the real model will be somewhat slower.

It runs two modes: BF16 activations with int8 weights, and int8 activations (W8A8), which uses the v5e's int8 units.

This needs only the converted folder. In a fresh session, run the settings and library cells first.

In [ ]:
import functools

import jax
import jax.numpy as jnp

print(jax.devices())
TEXT = json.loads((OUTPUT_DIR / "config.json").read_text())["text_config"]
INTERVAL = TEXT["full_attention_interval"]
assert all((kind == "full_attention") == ((layer + 1) % INTERVAL == 0) for layer, kind in enumerate(TEXT["layer_types"]))
BLOCKS = len(TEXT["layer_types"]) // INTERVAL
LINEAR_PER_BLOCK = INTERVAL - 1
CHUNK = 64
CAUSAL = jnp.tril(jnp.ones((CHUNK, CHUNK), dtype=bool))

weight_files = {
    file_name: SafetensorsFile(OUTPUT_DIR / file_name)
    for file_name in ("linear_attention.safetensors", "full_attention.safetensors", "mlp.safetensors", "layer_norms.safetensors")
}


def to_device(file_name, key, leading_shape):
    file = weight_files[file_name]
    array = jnp.asarray(np.ascontiguousarray(file.raw(key)))
    if file.header[key]["dtype"] == "BF16":
        array = jax.lax.bitcast_convert_type(array, jnp.bfloat16)
    return array.reshape(*leading_shape, *array.shape[1:])


def load_group(file_name, keys, leading_shape):
    return {key: to_device(file_name, key, leading_shape) for key in keys}


def int8_keys(projections):
    return [f"{projection}.{part}" for projection in projections for part in ("int8", "scale")]


blocks = {
    "linear": load_group("linear_attention.safetensors", int8_keys(["in_proj_qkv", "in_proj_z", "out_proj"]) + ["in_proj_a", "in_proj_b"], (BLOCKS, LINEAR_PER_BLOCK)),
    "full": load_group("full_attention.safetensors", int8_keys(["q_proj", "k_proj", "v_proj", "o_proj"]) + ["q_norm", "k_norm"], (BLOCKS,)),
    "mlp": load_group("mlp.safetensors", int8_keys(["gate_proj", "up_proj", "down_proj"]), (BLOCKS, INTERVAL)),
    "norms": load_group("layer_norms.safetensors", ["input_layernorm", "post_attention_layernorm"], (BLOCKS, INTERVAL)),
}
print(f"loaded {sum(array.nbytes for array in jax.tree.leaves(blocks)) / 1e9:.1f} GB onto {jax.devices()[0]}")


def int8_matmul(x, weights, projection, int8_activations):
    weight, scale = weights[f"{projection}.int8"], weights[f"{projection}.scale"]
    if int8_activations:
        x_scale = jnp.maximum(jnp.max(jnp.abs(x), axis=-1, keepdims=True).astype(jnp.float32), 1e-6) / 127
        x_int8 = jnp.clip(jnp.round(x / x_scale), -127, 127).astype(jnp.int8)
        product = jax.lax.dot(x_int8, weight, preferred_element_type=jnp.int32).astype(jnp.float32) * x_scale
    else:
        product = jnp.dot(x, weight.astype(jnp.bfloat16), preferred_element_type=jnp.float32)
    return (product * scale).astype(jnp.bfloat16)


def rms_norm(x, weight):
    x32 = x.astype(jnp.float32)
    return (x32 * jax.lax.rsqrt(jnp.mean(x32 * x32, axis=-1, keepdims=True) + TEXT["rms_norm_eps"]) * weight).astype(jnp.bfloat16)


def chunked_linear_attention(query, key, value, decay):
    """Gated DeltaNet prefill's shapes and serial chunk steps, without the delta-rule correction."""
    tokens, heads, key_dim = query.shape
    padding = (-tokens) % CHUNK

    def to_chunks(array):
        array = jnp.pad(array, [(0, padding)] + [(0, 0)] * (array.ndim - 1))
        return array.reshape(-1, CHUNK, *array.shape[1:])

    def step(state, chunk):
        chunk_query, chunk_key, chunk_value, chunk_decay = chunk
        scores = jnp.where(CAUSAL[None], jnp.einsum("lhd,mhd->hlm", chunk_query, chunk_key), 0)
        output = jnp.einsum("lhd,hdv->lhv", chunk_query, state) + jnp.einsum("hlm,mhv->lhv", scores, chunk_value)
        state = chunk_decay[:, None, None] * state + jnp.einsum("lhd,lhv->hdv", chunk_key, chunk_value)
        return state, output.astype(jnp.bfloat16)

    initial_state = jnp.zeros((heads, key_dim, value.shape[-1]), jnp.float32)
    chunks = (to_chunks(query), to_chunks(key), to_chunks(value), to_chunks(decay).mean(axis=1))
    _, outputs = jax.lax.scan(step, initial_state, chunks)
    return outputs.reshape(-1, heads, value.shape[-1])[:tokens]


def linear_attention_layer(x, weights, int8_activations):
    tokens = x.shape[0]
    key_heads, value_heads = TEXT["linear_num_key_heads"], TEXT["linear_num_value_heads"]
    key_dim, value_dim = TEXT["linear_key_head_dim"], TEXT["linear_value_head_dim"]
    qkv = int8_matmul(x, weights, "in_proj_qkv", int8_activations)
    query, key, value = jnp.split(qkv, [key_heads * key_dim, 2 * key_heads * key_dim], axis=-1)
    query = jnp.repeat(query.reshape(tokens, key_heads, key_dim), value_heads // key_heads, axis=1)
    key = jnp.repeat(key.reshape(tokens, key_heads, key_dim), value_heads // key_heads, axis=1)
    decay = jax.nn.sigmoid(jnp.dot(x, weights["in_proj_a"]).astype(jnp.float32))
    value = value.reshape(tokens, value_heads, value_dim) * jax.nn.sigmoid(jnp.dot(x, weights["in_proj_b"]))[..., None]
    mixed = chunked_linear_attention(query, key, value, decay).reshape(tokens, -1)
    gated = mixed * jax.nn.silu(int8_matmul(x, weights, "in_proj_z", int8_activations))
    return int8_matmul(gated, weights, "out_proj", int8_activations)


def full_attention_layer(x, weights, int8_activations):
    tokens = x.shape[0]
    heads, kv_heads, head_dim = TEXT["num_attention_heads"], TEXT["num_key_value_heads"], TEXT["head_dim"]
    query_and_gate = int8_matmul(x, weights, "q_proj", int8_activations).reshape(tokens, heads, 2 * head_dim)
    query, gate = jnp.split(query_and_gate, 2, axis=-1)
    key = int8_matmul(x, weights, "k_proj", int8_activations).reshape(tokens, kv_heads, head_dim)
    value = int8_matmul(x, weights, "v_proj", int8_activations).reshape(tokens, kv_heads, head_dim)
    query, key = rms_norm(query, weights["q_norm"]), rms_norm(key, weights["k_norm"])
    attended = jax.nn.dot_product_attention(query[None], key[None], value[None], is_causal=True)[0]
    gated = (attended * jax.nn.sigmoid(gate)).reshape(tokens, heads * head_dim)
    return int8_matmul(gated, weights, "o_proj", int8_activations)


def mlp(x, weights, int8_activations):
    gate = int8_matmul(x, weights, "gate_proj", int8_activations)
    up = int8_matmul(x, weights, "up_proj", int8_activations)
    return int8_matmul(jax.nn.silu(gate) * up, weights, "down_proj", int8_activations)


def layer_slice(group, position):
    return {key: array[position] for key, array in group.items()}


def block(x, weights, int8_activations):
    norms = weights["norms"]
    for position in range(INTERVAL):
        normed = rms_norm(x, norms["input_layernorm"][position])
        if position < LINEAR_PER_BLOCK:
            x = x + linear_attention_layer(normed, layer_slice(weights["linear"], position), int8_activations)
        else:
            x = x + full_attention_layer(normed, weights["full"], int8_activations)
        x = x + mlp(rms_norm(x, norms["post_attention_layernorm"][position]), layer_slice(weights["mlp"], position), int8_activations)
    return x


@functools.partial(jax.jit, static_argnames="int8_activations")
def proxy_forward(hidden, blocks, int8_activations):
    def scan_block(x, weights):
        return block(x, weights, int8_activations), None

    hidden, _ = jax.lax.scan(scan_block, hidden, blocks)
    return hidden

In [ ]:
BENCHMARK_TOKENS = 6820  # the T4 benchmark's request: 64 questions, 6,820 input tokens
BENCHMARK_RUNS = 10
T4_MEDIAN_MS = 8769  # llama.cpp, Q4_K_M GGUF on a Colab T4
V5E_PEAK_TFLOPS = {False: 197, True: 394}

hidden = (jax.random.normal(jax.random.key(0), (BENCHMARK_TOKENS, TEXT["hidden_size"])) * 0.02).astype(jnp.bfloat16)
int8_parameters = sum(array.size for group in blocks.values() for key, array in group.items() if key.endswith(".int8"))
matmul_tflop = 2 * BENCHMARK_TOKENS * int8_parameters / 1e12
print(f"projections and MLPs: {matmul_tflop:.1f} TFLOP per forward pass")

for int8_activations in (False, True):
    label = "int8 activations (W8A8)" if int8_activations else "BF16 activations, int8 weights"
    started = time.perf_counter()
    compiled = proxy_forward.lower(hidden, blocks, int8_activations=int8_activations).compile()
    memory = compiled.memory_analysis()
    print(f"{label}: compiled in {time.perf_counter() - started:.0f}s, working memory {memory.temp_size_in_bytes / 1e9:.2f} GB")

    compiled(hidden, blocks).block_until_ready()
    latencies_ms = []
    for _ in range(BENCHMARK_RUNS):
        started = time.perf_counter()
        compiled(hidden, blocks).block_until_ready()
        latencies_ms.append((time.perf_counter() - started) * 1000)
    median_ms = statistics.median(latencies_ms)
    p95_ms = statistics.quantiles(latencies_ms, n=20, method="inclusive")[-1]
    print(
        f"  {BENCHMARK_TOKENS} tokens, {BENCHMARK_RUNS} runs: median {median_ms:.0f} ms | p95 {p95_ms:.0f} ms | "
        f"min {min(latencies_ms):.0f} ms | {BENCHMARK_TOKENS / (median_ms / 1000):.0f} tokens/s | {T4_MEDIAN_MS / median_ms:.1f}x the T4"
    )
    print(f"  matmul throughput {matmul_tflop / (median_ms / 1000):.0f} of {V5E_PEAK_TFLOPS[int8_activations]} peak TFLOP/s")

print(jax.devices()[0].memory_stats())

## Notes

- **The output disappears when the runtime ends**, so run the upload cell before stopping. In a later session, log in (the repo is private) and run `hf download <your-username>/clef-flash-tpu-int8 --local-dir /content/clef-flash-tpu`; there's no need to download the original or convert again. If the shell can't find `hf`, use the path that cell 2 prints.
- **Norm weights are stored exactly as released.** The runtime applies them the way transformers' `Qwen3_5` code does: `(1 + weight)` for the RMSNorms, plain `weight` for the DeltaNet output norm.
- **The int8 is weights only.** Quantizing activations as well (W8A8) is a runtime choice with its own accuracy check.
- **To run the model:** `hub/clef_tpu.py` loads this output on the TPU and answers `/v1/systemone` requests; `hub/notebook.ipynb` serves it over HTTP.